# Challenge: Walk Forward on Other Datasets

## Download data from `yfinance`

In [12]:
import yfinance as yf

ticker = 'AAPL'
df = yf.download(ticker, multi_level_index=False, auto_adjust=False)
df

[*********************100%***********************]  1 of 1 completed


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-04-30,271.100250,271.350006,276.000000,268.140015,270.500000,91848200
2026-05-01,279.882141,280.140015,287.220001,278.369995,278.859985,79915400
2026-05-04,276.575165,276.829987,280.630005,274.859985,279.660004,46668400
2026-05-05,283.918427,284.179993,284.570007,276.500000,276.929993,49311700
2026-05-06,287.245361,287.510010,288.029999,281.070007,281.920013,58336100
2026-05-07,287.175415,287.440002,292.130005,285.779999,289.269989,45224300
2026-05-08,293.050018,293.320007,294.760010,290.000000,290.010010,52692800
2026-05-11,292.679993,292.679993,293.880005,290.230011,291.980011,42247300
2026-05-12,294.799988,294.799988,295.269989,292.559998,292.559998,45748100


## Preprocess the data

### Filter the date range

In [13]:
df = df.loc['2018-01-01':].copy()

### Create the target variable

#### Percentage change

- Percentage change on `Adj Close` for tomorrow

In [14]:
df['change_tomorrow'] = df['Adj Close'].pct_change(-1)
df.change_tomorrow = df.change_tomorrow * -1
df.change_tomorrow = df.change_tomorrow * 100

#### Remove rows with any missing data

In [15]:
df = df.dropna().copy()
df

,Adj Close,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,,
2026-04-30,271.100250,271.350006,276.000000,268.140015,270.500000,91848200,3.137710
2026-05-01,279.882141,280.140015,287.220001,278.369995,278.859985,79915400,-1.195688
2026-05-04,276.575165,276.829987,280.630005,274.859985,279.660004,46668400,2.586398
2026-05-05,283.918427,284.179993,284.570007,276.500000,276.929993,49311700,1.158221
2026-05-06,287.245361,287.510010,288.029999,281.070007,281.920013,58336100,-0.024357
2026-05-07,287.175415,287.440002,292.130005,285.779999,289.269989,45224300,2.004642
2026-05-08,293.050018,293.320007,294.760010,290.000000,290.010010,52692800,-0.126427
2026-05-11,292.679993,292.679993,293.880005,290.230011,291.980011,42247300,0.719130
2026-05-12,294.799988,294.799988,295.269989,292.559998,292.559998,45748100,1.361799


## Machine Learning modelling

### Separate the data

1. Target: which variable do you want to predict?
2. Explanatory: which variables will you use to calculate the prediction?

In [16]:
y = df.change_tomorrow
X = df[['Open','High','Low','Close','Volume']]

### Time Series Split

In [17]:
from sklearn.model_selection import TimeSeriesSplit

In [18]:
ts = TimeSeriesSplit

### Compute and evaluate model in a for loop

1. Separate the data in train and test
2. Compute the model on the train set
3. Evaluate the model (mse) on the test set
4. Append the errors (mse) in an empty list

In [30]:
from sklearn.metrics import mean_squared_error

In [31]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model_dt = RandomForestRegressor(max_depth=15, random_state=42)

error_mse_list = []

for index_train, index_test in ts.split(df):
    X_train, y_train = X.iloc[index_train], y.iloc[index_train]
    X_test, y_test = X.iloc[index_test], y.iloc[index_test]
    
    model_dt.fit(X_train, y_train)
    
    y_pred = model_dt.predict(X_test)
    error_mse = mean_squared_error(y_test, y_pred)
    
    error_mse_list.append(error_mse)

TypeError: split() missing 1 required positional argument: 'X'

In [32]:
error_mse_list

[]

## Anchored Walk Forward evaluation in backtesting

![](<src/10_Table_Validation Methods.png>)

### Create a new strategy

In [24]:
%load_ext autoreload
%autoreload 2

In [25]:
from backtesting import Strategy

In [33]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    
    n_train = 600
    coef_retrain = 200
    
    def init(self):
        self.model = RandomForestRegressor(max_depth=15, random_state=42)
        self.already_bought = False
        
        X_train = self.data.df.iloc[:self.n_train, :-1]
        y_train = self.data.df.iloc[:self.n_train, -1]
        
        self.model.fit(X=X_train, y=y_train)

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :-1]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

In [34]:
class WalkForwardAnchored(Regression):
    def next(self):
        
        # we don't take any action and move on to the following day
        if len(self.data) < self.n_train:
            return
        
        # we retrain the model each 200 days
        if len(self.data) % self.coef_retrain == 0:
            X_train = self.data.df.iloc[:, :-1]
            y_train = self.data.df.iloc[:, -1]

            self.model.fit(X_train, y_train)

            super().next()
            
        else:
            
            super().next()

### Run the backtest with optimization

In [35]:
import multiprocessing as mp
mp.set_start_method('fork', Force= True)

TypeError: set_start_method() got an unexpected keyword argument 'Force'

In [ ]:
stats_skopt, heatmap, optimize_result = bt.optimize(
    limit_buy = range(0, 6), limit_sell = range(-6, 0),
    maximize='Return [%]',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True,
    method='skopt'
    )

dff = heatmap.reset_index()
dff = dff.sort_values('Return [%]', ascending=False)
dff

In [ ]:
bt.optimize(???)

## Unanchored Walk Forward

### Create a library of strategies

[strategies.py](strategies.py)

### Create the unanchored walk forward class

In the previously created library

![](<src/10_Table_Validation Methods.png>)

### Import the strategy and perform the backtest with optimization

In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
import strategies

In [28]:
strategies.WalkForwardUnanchored

strategies.WalkForwardUnanchored

In [29]:
bt_unanchored = Backtest(df, strategies.WalkForwardUnanchored, cash=10000, commission=.002, exclusive_orders=True)

stats_skopt, heatmap, optimize_result = bt_unanchored.optimize(
    limit_buy = range(0, 6), limit_sell = range(-6, 0),
    maximize='Return [%]',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True,
    method='skopt'
    )

dff = heatmap.reset_index()
dff = dff.sort_values('Return [%]', ascending=False)
dff


NameError: name 'Backtest' is not defined

### Interpret the strategies' performance

In [ ]:
bt.???

In [ ]:
bt_unanchored.???

## Course Conclusion

Watch video → [Next steps]()